# Ouro-1.4B-Thinking：BF16 / NVFP4 PTQ 基线

本地修改 → `python scripts/package_for_colab.py` → 上传新 ZIP → 在本 notebook 比较。
固定 Thinking checkpoint 和测试集版本；本地源码直接执行。**先选原生支持 BF16 的 GPU，例如 L4 / A100；T4 不适用于 BF16 对照。**

当前流程：MATH-500 分成互不重叠的 5 批，每批 100 题。每次只运行一批，完成后手动切换 `BATCH_ID=1..5`。
每批 100 题 × BF16/PTQ × 两种阈值 = 400 次生成；五批共 2000 次生成，最多 2048 新 tokens。
默认把缓存和逐题结果保存到 Google Drive；五批完成后用第 11、12 节合并并下载完整 500 题统计。

- BF16：原始权重；PTQ：仅核心 MLP 的 NVFP4 W4A16 RTN 数值模拟，其余保持 BF16。
- PTQ 权重反量化后仍用 BF16 存储/计算，不是原生 4-bit kernel，不能据此比较压缩或加速。
- 两阈值独立生成、各自使用新缓存。阈值 1.0：固定 4 轮；0<threshold<1：累计退出概率达到阈值就停止，最多 4 轮。
- 同一份模型和生成代码保留两种行为。默认 `ENABLE_EARLY_EXIT=True`：真实提前退出，选中深度等于执行深度。跳过轮次的当前 token KV 用最后执行轮的对应层 KV 补齐；后续输出可能与旧模拟退出不同。
- `ENABLE_EARLY_EXIT=False`：跑满 4 轮后按阈值选择输出。两种模式下 threshold=1 都固定跑满。
- 结果目录自动带 early-exit 或 simulated；两种模式独立统计，便于对比。
- 这是单次贪心作答准确率，不是多样本 pass@k，也不直接等于论文中的评测设置。


项目目录：`ouro-loopedlm-experiments`。新实验统一执行 `python scripts/package_for_colab.py` 打包。

继续已完成前 200 题的 MATH-500 实验：上传冻结的 `ouro-1.4b-colab-a36a55649dbb.zip`，第 6 节保持 `ENABLE_EARLY_EXIT=False`；详情见 `docs/project-layout.md`。


## 1. 选择 GPU
运行时 → 更改运行时类型 → 原生 BF16 GPU（例如 L4 / A100）。

In [ ]:
import sys
import subprocess
import torch
from packaging.version import Version

if sys.version_info < (3, 10):
    raise RuntimeError("This notebook requires Python 3.10 or newer.")
if not (Version("2.6") <= Version(torch.__version__.split("+")[0]) < Version("3")):
    raise RuntimeError("Use a Colab runtime with PyTorch >= 2.6 and < 3.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a GPU runtime, then reconnect.")
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError("Native BF16 required. T4 cannot run the BF16 baseline; select L4/A100 or another BF16 GPU.")


## 2. 上传本地源码 ZIP

选择本地 `dist/` 中最新的 `ouro-1.4b-colab-xxxxxxxxxxxx.zip`。
每次上传均解压到新目录；包内哈希清单用来发现传输错误。
输出中的 `modeling_ouro.py SHA256` 应与你本地修改后的文件一致。
只上传你自己打包的项目，因为接下来会执行包内代码。

In [ ]:
from google.colab import files
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one project ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("Upload the ZIP created by scripts/package_for_colab.py.")
workspace = Path(os.environ.get("OURO_COLAB_WORKSPACE", "/content"))
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = archive.infolist()
    if len({m.filename for m in members}) != len(members):
        raise ValueError("Duplicate ZIP paths.")
    if sum(m.file_size for m in members) > 100 * 1024 * 1024:
        raise ValueError("Source bundle is unexpectedly large; exclude weights and caches.")
    roots = {PurePosixPath(m.filename).parts[0] for m in members if PurePosixPath(m.filename).parts}
    if len(roots) != 1 or not roots <= {"ouro-loopedlm-experiments", "ouro-1.4b-colab"}:
        raise ValueError("Expected a unified project ZIP or a frozen historical baseline ZIP.")
    archive_root = next(iter(roots))
    for member in members:
        path = PurePosixPath(member.filename)
        if (path.is_absolute() or ".." in path.parts or "\\" in member.filename
                or not path.parts or path.parts[0] != archive_root
                or (member.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError("Unsafe ZIP path: " + member.filename)
    prefix = archive_root + "/"
    manifest_name = prefix + "bundle_manifest.json"
    manifest = json.loads(archive.read(manifest_name))
    if manifest.get("format") != 1:
        raise ValueError("Unsupported source bundle format.")
    expected = {prefix + name for name in manifest["files"]} | {manifest_name}
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != expected:
        raise ValueError("ZIP contents do not match the source manifest.")
    for relative, entry in manifest["files"].items():
        data = archive.read(prefix + relative)
        if hashlib.sha256(data).hexdigest() != entry["sha256"] or len(data) != entry["bytes"]:
            raise ValueError("SHA256 hash mismatch: " + relative)
    run_dir = Path(tempfile.mkdtemp(prefix="ouro-source-", dir=workspace))
    for member in members:
        if not member.is_dir():
            target = run_dir.joinpath(*PurePosixPath(member.filename).parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))

PROJECT_DIR = run_dir / archive_root
print("Project directory:", PROJECT_DIR)
print("Verified source files:", len(manifest["files"]))
print("modeling_ouro.py SHA256:", manifest["files"]["ouro_local/modeling_ouro.py"]["sha256"])

## 3. 安装依赖
保留 Colab 自带 CUDA PyTorch。安装与执行均使用子进程；首次或依赖变动后运行。

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT_DIR / "requirements-bench.txt")], check=True)

## 4. 缓存和结果存储
默认 `USE_GOOGLE_DRIVE=True`，把下载缓存和每批结果保存在 Drive；首次挂载由 Colab 请求授权。
也可以设为 False 使用临时磁盘，但必须在运行时删除前下载每批结果；保存 notebook 不会保存这些结果。
断点续跑需相同源码、参数、数据及运行环境，并设置同一个 `BATCH_ID`、`RUN_PREFIX` 和 `RESUME=True`。
五批使用同一源码 ZIP、seed、GPU 型号及依赖版本；合并时会检查一致性。


In [ ]:
import os
from pathlib import Path
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ouro-experiments")
else:
    STORAGE_ROOT = Path("/content/ouro-experiments")
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(STORAGE_ROOT / "hf-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUTF8"] = "1"
print("Cache and results:", STORAGE_ROOT)

## 5. 先检查本地源码
仅随机小模型，不下载完整权重。

In [ ]:
subprocess.run([sys.executable, str(PROJECT_DIR / "scripts/smoke_test.py")], cwd=PROJECT_DIR, check=True)
for pattern in ("test_baselines.py", "test_batches.py", "test_early_exit.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
                   cwd=PROJECT_DIR, check=True)


## 6. 实验参数：选择本次批次
固定 seed 打乱一次 MATH-500，再按位置切分；下面的位置不是原始题目 ID。

| BATCH_ID | OFFSET | 打乱后的位置 | 题数 |
|---|---|---|---|
| 1 | 0 | 1–100 | 100 |
| 2 | 100 | 101–200 | 100 |
| 3 | 200 | 201–300 | 100 |
| 4 | 300 | 301–400 | 100 |
| 5 | 400 | 401–500 | 100 |

每批完成全部 BF16/PTQ 和两种阈值后，再切换下一批；不要通过更换 seed 来分批。
开始下一批：只改 `BATCH_ID`，保持 `RESUME=False`，依次运行第 6 → 7 → 8 → 10 节（第 9 节可选）。
恢复中断批次：保持原 `BATCH_ID`、`RUN_PREFIX`、全部设置和环境，改为 `RESUME=True` 后运行第 6、7 节。
新 Colab 会话从第 1 节重新准备；本地 notebook 修改不会自动同步到已打开的 Colab。
`LOOPS=None` 仍沿用本地原始 4-loop 配置；生成预算、prompt、seed、BF16/PTQ 实现均保持不变。
只需修改 `ENABLE_EARLY_EXIT`：True 为真实退出，False 为跑满后选择；EXECUTION_MODE 和 RUN_PREFIX 自动生成。
对比同一批时保持 BATCH_ID/seed/题数/阈值/预算，切换开关并设 RESUME=False，重跑第 6–10 节。
每种模式的五批保持相同开关和源码；合并前在第 6 节选回该模式。不同模式各自合并，再并排比较。
所有批次完成后才运行第 11 节；不完整批次会被合并器拒绝，应先续跑完成。


In [ ]:
DATASETS = ["math500"]
LIMIT = 100
BATCH_ID = 1  # 每次运行一批；依次改成 1、2、3、4、5
ENABLE_EARLY_EXIT = True  # True: 达到阈值即停；False: 跑满后按阈值选输出
if not isinstance(ENABLE_EARLY_EXIT, bool):
    raise ValueError("ENABLE_EARLY_EXIT must be True or False.")
EXECUTION_MODE = "early_exit" if ENABLE_EARLY_EXIT else "simulated"
RUN_PREFIX = f"baseline-math500-100q-{EXECUTION_MODE.replace('_', '-')}"
THRESHOLDS = [1.0, 0.5]
LOOPS = None
MAX_NEW_TOKENS = 2048
MAX_TOTAL_TOKENS = 4096
SEED = 42
RESUME = False  # 仅恢复同一批时改为 True；开始下一批改回 False
if not isinstance(BATCH_ID, int) or not 1 <= BATCH_ID <= 5 or LIMIT < 1:
    raise ValueError("Select BATCH_ID from 1 to 5 and a positive per-batch LIMIT.")
OFFSET = (BATCH_ID - 1) * LIMIT
RUN_NAME = f"{RUN_PREFIX}-batch-{BATCH_ID:02d}"
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in (".", ".."):
    raise ValueError("RUN_NAME must be a simple directory name.")
RESULT_DIR = STORAGE_ROOT / "results" / RUN_NAME
print("Early exit enabled:", ENABLE_EARLY_EXIT, "mode:", EXECUTION_MODE)
print("Batch:", BATCH_ID, "offset:", OFFSET, "limit per dataset:", LIMIT)
print("Result directory:", RESULT_DIR)
print("Approximate number of responses:", 2 * len(THRESHOLDS) * len(DATASETS) * LIMIT)


## 7. 运行两组基线
先跑 BF16，再原地量化 MLP 后跑 PTQ；GPU 中只加载一份模型。
每个阈值从同一问题独立生成自己的完整轨迹，缓存不共用。
自适应模式先用满深度处理 prompt 的前 P−1 个 token，再对最后 prompt token 和后续生成逐次提前退出。
每完成一题立即保存，日志显示答对情况、token 数、gate 深度、实际执行深度和停止原因。
完整模型首次会下载约 2.87 GB 权重。源码或实验设置变化后必须使用新结果目录。

In [ ]:
command = [sys.executable, str(PROJECT_DIR / "benchmark.py"),
           "--datasets", *DATASETS, "--limit", str(LIMIT), "--offset", str(OFFSET),
           "--thresholds", *map(str, THRESHOLDS), "--seed", str(SEED),
           "--execution-mode", EXECUTION_MODE,
           "--max-new-tokens", str(MAX_NEW_TOKENS),
           "--max-total-tokens", str(MAX_TOTAL_TOKENS), "--output-dir", str(RESULT_DIR)]
if LOOPS is not None:
    command.extend(["--loops", str(LOOPS)])
if RESUME:
    command.append("--resume")
print("Running:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

## 8. 查看比较
- `accuracy`：最终答案评分；`parse_failure_rate`：未能解析最终答案的比例。
- `mean_generated_tokens`：不含 prompt，包含生成的 reasoning 和终止 EOS。
- `mean_selected_depth`：按生成 token 加权的 gate 选择深度。
- `mean_executed_depth`：每个生成决策实际执行的循环数；阈值 1 固定为 4，阈值 0.5 在满足条件时小于 4。
- `mean_executed_core_steps`：每题生成决策的循环数之和 C；真实退出模式等于 `mean_selected_core_steps`。
- `execution_mode`：开关 True 对应 early_exit，False 对应 simulated；模拟模式的 executed 固定为 4，selected 深度仅是代理。
- 计算量还可查看 JSONL 中 `actual_forward_token_loops`：4×(prompt_tokens−1)+Σ executed depth。循环计数不等于 FLOPs 或实测加速倍数。
- `truncation_rate`：达到长度上限而未遇到 EOS。截断题照常计入分母，原始回答可复查。

配对摘要含双方都答对题目的计算膨胀比例和逐题标准误。没有共同答对题目时显示 null，不能当作 0。
本节展示本批 100 题；完整 500 题统计在第 11 节重新计算。不要直接平均五份摘要里的深度、标准误或计算膨胀比例。
评分使用保留特殊 token 的原始输出，只读取 `</think>` 后的最终回答；thinking 尚未闭合记为解析失败。


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
summary_path = RESULT_DIR / "summary.json"
if not summary_path.exists():
    raise RuntimeError("Run the experiment first; partial summaries appear after the first completed problem.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
table = pd.read_csv(RESULT_DIR / "summary.csv")
display(table)
for dataset in table["dataset"].unique():
    subset = table[table["dataset"] == dataset].copy()
    subset["label"] = subset["arm"] + " / threshold=" + subset["threshold"].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    for ax, metric, title in zip(axes, ["accuracy", "mean_generated_tokens", "mean_executed_depth"],
                                ["Accuracy", "Generated tokens", "Executed core loops per token"]):
        ax.bar(subset["label"], subset[metric])
        ax.set_title(f"{dataset}: {title}")
        ax.tick_params(axis="x", labelrotation=40)
        if metric == "accuracy": ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()
print("Paired comparisons:")
print(json.dumps(summary["paired"], ensure_ascii=False, indent=2))

## 9. 查看一题的完整输出和深度
可以修改 `RECORD_INDEX`；每个 token 的 ID、gate 概率与深度都在 JSONL 中。`exit_pdfs` 中 null 表示该轮没有执行；不是零概率。

In [ ]:
RECORD_INDEX = 0
records = [json.loads(line) for line in (RESULT_DIR / "results.jsonl").read_text(encoding="utf-8").splitlines()]
record = records[RECORD_INDEX]
print({k: record[k] for k in ["dataset", "example_id", "arm", "threshold", "correct", "extracted_answer", "stop_reason"]})
print("Question:", record["question"])
print("Gold:", record["gold_answer"])
print(record["generated_text_raw"])
plt.figure(figsize=(12, 3))
plt.step(range(1, len(record["selected_depths"]) + 1), record["selected_depths"], label="gate selected", where="mid")
plt.plot(range(1, len(record["executed_depths"]) + 1), record["executed_depths"], label="executed", alpha=.6)
plt.xlabel("Generated token index (1-based)")
plt.ylabel("Core loops")
plt.legend()
plt.tight_layout()
plt.show()

## 10. 下载结果
包含 manifest、逐题原始回答/深度、量化范围清单和 CSV/JSON 摘要；中断时也可下载已完成部分。

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(RESULT_DIR.parent / (RESULT_DIR.name + "-results")), "zip", root_dir=RESULT_DIR)
files.download(archive)

## 11. 五批全部完成后：合并完整 MATH-500 结果
先在第 6 节选回要合并的 ENABLE_EARLY_EXIT 并运行参数单元格，以确定 RUN_PREFIX。只在该模式五批均完成 400 次生成后运行。合并不需要模型推理，通常很快。
读取五个结果目录的原始 JSONL；检查同一实验协议、每批四种组合齐全、没有重复题，恰好覆盖 500 个题目 ID。
重新计算准确率、token 数、token 加权深度、截断率、标准误及双方答对题上的配对指标，不读取各批摘要作为计算输入。
输出保留每批的 manifest 和量化清单；不会覆盖已有的合并目录。
如果在本地合并下载的五个 ZIP，可以用 `python scripts/merge_results.py --inputs 批1.zip 批2.zip 批3.zip 批4.zip 批5.zip --output-dir outputs/math500-full`，只需要 Python 标准库。


In [ ]:
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# 也可改成五个已上传的结果 ZIP 路径；默认从 Drive 的五个批次目录读取。
MERGE_INPUTS = [STORAGE_ROOT / "results" / f"{RUN_PREFIX}-batch-{i:02d}" for i in range(1, 6)]
MERGED_DIR = STORAGE_ROOT / "results" / f"{RUN_PREFIX}-merged-500"
merge_command = [sys.executable, str(PROJECT_DIR / "scripts/merge_results.py"),
                 "--inputs", *map(str, MERGE_INPUTS), "--dataset", "math500",
                 "--expected-examples", "500", "--output-dir", str(MERGED_DIR)]
subprocess.run(merge_command, cwd=PROJECT_DIR, check=True)
display(pd.read_csv(MERGED_DIR / "summary.csv"))
merged_summary = json.loads((MERGED_DIR / "summary.json").read_text(encoding="utf-8"))
print("Full 500-question paired comparisons:")
print(json.dumps(merged_summary["paired"], ensure_ascii=False, indent=2))


## 12. 下载完整 500 题合并结果
包含 2000 条逐次生成记录、重新计算的 CSV/JSON 摘要和五批原始实验配置。原始批次目录保持不变。


In [ ]:
import shutil
from google.colab import files
merged_archive = shutil.make_archive(str(MERGED_DIR.parent / (MERGED_DIR.name + "-results")), "zip", root_dir=MERGED_DIR)
files.download(merged_archive)
